In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Library

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# Utils

In [4]:
# Top 5 core Filtering
def top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자 필터링
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        data = data[~data['user_id'].isin(users_to_remove)]

        # 아이템 필터링
        item_counts = data['asin'].value_counts()
        items_to_remove = item_counts[item_counts < chk].index
        data = data[~data['asin'].isin(items_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        # 두 조건 모두 만족 시 종료
        if (len(users_to_remove) == 0) and (len(items_to_remove) == 0):
            break

        cnt += 1

    return data

#  Sample Clothing and Shoes

In [3]:
# 리뷰 데이터셋 확인
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_final.pkl')

print(df.shape)
df.head()

(66718, 17)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb,LLaMA_Output,LLaVA_Output
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.20688048, -0.32638466, 0.13180715, -0.5702...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06857138, 0.13921358, -0.039587017, -0.088...","Based on user reviews, it appears that users a...",The image shows a hand holding a set of earrin...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.062450625, -0.07075825, 0.15572476, -0.2797...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.04617965, 0.12618569, -0.017711096, -0.102...","Based on user reviews, it appears that users p...",The image shows a close-up of a person's ear w...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.26902932, -0.42407188, 0.82339007, -0.2940...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.062135138, 0.1333665, -0.019405615, -0.102...","Based on this review, it seems that users pref...","The image shows a fashion accessory, specifica..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.119323395, -0.57359934, 0.40760195, -0.2408...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06780286, 0.051634714, -0.017799914, -0.11...","Based on user reviews, it appears that users p...",The image shows a sleeveless tank top with a p...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.43734312, -0.62041384, 0.14694075, -0.3723...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.07941287, 0.05176169, -0.02554227, -0.0903...","Based on the user review, it appears that the ...",The image shows a lime green t-shirt with a gr...


In [16]:
df_sample = df.sample(n=49000, random_state=42)
df_sample = top_n_filtering(df_sample, chk=3)
df_sample = df_sample.reset_index(drop=True)

print(df_sample.shape)
df_sample.head()

Epoch 1
 - Remaining users: 5936
 - Remaining items: 9055
 - Remaining reviews: 34856
Epoch 2
 - Remaining users: 4735
 - Remaining items: 8156
 - Remaining reviews: 30981
Epoch 3
 - Remaining users: 4506
 - Remaining items: 7956
 - Remaining reviews: 30149
Epoch 4
 - Remaining users: 4445
 - Remaining items: 7912
 - Remaining reviews: 29940
Epoch 5
 - Remaining users: 4434
 - Remaining items: 7903
 - Remaining reviews: 29900
Epoch 6
 - Remaining users: 4429
 - Remaining items: 7898
 - Remaining reviews: 29880
Epoch 7
 - Remaining users: 4426
 - Remaining items: 7895
 - Remaining reviews: 29868
Epoch 8
 - Remaining users: 4424
 - Remaining items: 7894
 - Remaining reviews: 29862
Epoch 9
 - Remaining users: 4424
 - Remaining items: 7894
 - Remaining reviews: 29862
(29862, 17)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb,LLaMA_Output,LLaVA_Output
0,AHNGXSJ44HVTEJAF4G6XHN4MJA7Q,B081SHSHMW,B081SHSHMW,"Price wise, these were definitely worth it!!<b...",5,Teardrop Hoop Earrings Sterling Silver -Cubic ...,"[-0.1002061591, 0.1200838983, -0.0344410017, -...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 19.7521076202, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.07341298, -0.4078031, 0.22591048, -0.04094...","[-0.2199544, -0.45286068, 0.25211662, -0.01336...","[-0.060513172, 0.096124716, -0.026089791, -0.1...","[-0.03661478, 0.123107545, -0.0307951, -0.1170...","Based on user reviews, it appears that custome...",The image shows a pair of earrings with a tear...
1,AF6TUIEFUW4ZPAZBE4QXCKO3HO3Q,B0BRRZBTS7,B0BRS2KCJN,Beautiful gold necklace! Subtle elegance that...,5,Awvialy Cute Heart Necklace Dainty Gold Heart ...,"[-0.1078577265, 0.1349532306, -0.0327288359, -...",https://m.media-amazon.com/images/I/61W-A6D8-7...,https://m.media-amazon.com/images/I/61W-A6D8-7...,https://m.media-amazon.com/images/I/61W-A6D8-7...,"[0.0, 0.7265028954, 0.0, 0.0, 0.0, 0.0, 0.0, 1...","[-0.0930238, -0.62754226, 0.57404345, -0.35024...","[-0.13979189, -0.5576768, 0.07676179, -0.50724...","[-0.05415082, 0.093080476, -0.022917453, -0.12...","[-0.069505736, 0.13157515, -0.035399992, -0.08...","Based on user reviews, it appears that users p...",The image shows a woman wearing a necklace wit...
2,AH3ETXH5DP245PTXW2SAIYP2333A,B0B4DBLP9B,B0BR5CTCPB,EXCEEDED MY EXPECTATIONS WITH IT'S MULTI FEATU...,5,shifenmei Digital Watch - Waterproof Digital S...,"[-0.0416909307, 0.056715101000000004, -0.04312...",https://m.media-amazon.com/images/I/61aZfew4z3...,https://m.media-amazon.com/images/I/61aZfew4z3...,https://m.media-amazon.com/images/I/61aZfew4z3...,"[0.0, 6.9544606209, 0.0, 0.0, 0.0, 0.0, 0.0, 0...","[-0.5039386, -0.24933721, 0.39972943, -0.35781...","[-0.3946679, -0.39823705, 0.59443426, -0.21039...","[-0.059465233, 0.010808255, -0.070039555, -0.1...","[-0.107101284, 0.11598985, -0.021455176, -0.12...","Based on user reviews, it appears that users h...",The image shows a digital watch with a black s...
3,AGE3JQYTMBQARXIVWFQG62YSHOSQ,B091LDJ3JY,B091LBJYRT,Design and colors look nice. Material is heavi...,3,Marmot Men's Culebra Peak Short-Sleeve T-Shirt,"[-0.1116923019, 0.0933089554, -0.0008338508000...",https://m.media-amazon.com/images/I/71RRYESEEJ...,https://m.media-amazon.com/images/I/71RRYESEEJ...,https://m.media-amazon.com/images/I/71RRYESEEJ...,"[0.0, 0.0, 0.0, 0.0, 20.5194454193, 0.0, 0.0, ...","[0.1600187, -0.4006862, 0.34607992, -0.1141110...","[0.03943097, -0.79769313, 0.22503811, -0.45568...","[-0.06992936, 0.095793225, -0.019993845, -0.10...","[-0.09232843, 0.11063729, -0.019075153, -0.093...","Based on user reviews, it appears that users a...",The image shows a man wearing a dark blue T-sh...
4,AH4VLBW6VFOLM2DRG557ZHVCL7YQ,B0BGQXWG3P,B0BGQYSJZF,Nice pair of earrings that you could wear with...,5,DELLA MODA Long Bar Dangle Earrings for Women ...,"[-0.0660392866, 0.137768954, -0.0492942072, -0...",https://m.media-amazon.com/images/I/71GWenu9TK...,https://m.media-amazon.com/images/I/71GWenu9TK...,https://m.media-amazon.com/images/I/71GWenu9TK...,"[0.0, 22.6954956055, 0.0, 3.6710858345, 0.2765...","[0.05159018, -0.87716293, 0.47148463, -0.35939...","[0.14315477, -0.025678355, 0.47234574, -0.4629...","[-0.07013296, 0.07829906, -0.022329235, -0.115...","[-0.06771848, 0.11242156, -0.030980976, -0.106...","Based on user reviews, it appears that users a...",The image shows a close-up of a person's ear w...


In [17]:
df_sample.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_Aspect.pkl')

In [18]:
df_sample = df_sample.drop(columns=['LLaMA_Output', 'LLaVA_Output'])
df_sample.columns

Index(['user_id', 'asin', 'parent_asin', 'text', 'rating', 'title_y',
       'roberta_emb', 'small_image_url', 'medium_image_url', 'large_image_url',
       'VGG16_emb', 'user_bert_emb', 'item_bert_emb', 'user_roberta_emb',
       'item_roberta_emb'],
      dtype='object')

In [19]:
df_sample.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_RQ3.pkl')